# How to extract a paragraph from a pdf file and store its position?

**Problem Statement:** I'm going to extract the content of a PDF file using PDFBox library. The content should be processed paragraph-by-paragraph and for each paragraph, I need its position for follow-up processing. Using the following code, I can extract the whole content of an input PDF: PDDocument doc = PDDocument.load(file); PDFTextStripper stripper = new PDFTextStripper(); S Generic engineering direction: Build a reproducible benchmark comparing baseline vs. structured mitigation.

**Referenced Discussions:**
- [stackoverflow] How to extract a paragraph from a pdf file and store its position? (https://stackoverflow.com/questions/25109969/how-to-extract-a-paragraph-from-a-pdf-file-and-store-its-position)


In [ ]:
# Reproducibility & Structured Metrics Helper (100% Interactive Colab — No Tunnels/Servers)
import json, platform, sys, subprocess

def log_lab_environment(extra=None):
    gpu = "CPU only"
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
        if out:
            gpu = out.splitlines()[0]
    except Exception:
        pass
    env_info = {
        "pythonVersion": platform.python_version(),
        "hardware": f"{gpu} ({platform.machine()})",
        "runtime": "Google Colab" if "google.colab" in sys.modules else platform.platform(),
    }
    if isinstance(extra, dict):
        env_info.update(extra)
    print("[LAB_ENV_JSON]", json.dumps(env_info))
    return env_info

def log_lab_result(metrics: dict):
    """Print structured metrics so the Lab workbench can import only compact numbers."""
    print("[LAB_RESULT_JSON]", json.dumps(metrics))

log_lab_environment()


In [ ]:
# Sort blocks by vertical position (y0), then horizontal (x0)
import fitz  # PyMuPDF

def extract_paragraphs(pdf_path, vertical_threshold=5.0):
    """
    Extracts text blocks and merges them if they are vertically proximal.
    """
    doc = fitz.open(pdf_path)
    paragraphs = []

    for page in doc:
        # Sort blocks by vertical position (y0), then horizontal (x0)
        blocks = sorted(page.get_text("blocks"), key=lambda b: (b[1], b[0]))
        
        current_para = None
        
        for b in blocks:
            x0, y0, x1, y1, text, block_no, block_type = b
            
            # Skip non-text blocks
            if block_type != 0: continue
            
            # Logic: If current_para exists AND gap between 
            # previous y1 and current y0 is small, merge.
            if current_para and (y0 - current_para['y1'] < vertical_threshold):
                current_para['text'] += " " + text.strip()
                current_para['y1'] = max(current_para['y1'], y1)
            else:
                # Start new paragraph
                if current_para: paragraphs.append(current_para)
                current_para = {'text': text.strip(), 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1}
        
        if current_para: paragraphs.append(current_para)
    
    return paragraphs